## rag(retrival augmented generation) from scratch

### ingestion pipeline

In [1]:
# loading text/data from PDF files
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

project_root = Path.cwd()
if not (project_root / "data").is_dir() and (project_root.parent / "data").is_dir():
    project_root = project_root.parent
data_dir = project_root / "data"
dir_loader = DirectoryLoader(str(data_dir), glob="**/*.pdf", loader_cls=PyPDFLoader)
documents = dir_loader.load()
print(f"Loaded {len(documents)} PDF pages from {data_dir}")


/tmp/ipykernel_12128/1617310034.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/UIZZEM+Corbel-Bold', '/FontDescriptor': IndirectObject(2481, 0, 128079786070320), '/Encoding': '/MacRomanEncoding', '/FirstChar': 32, '/LastChar': 121, '/Widths': [206, 0, 0, 0, 0, 0, 0, 0, 306, 306, 0, 0, 0, 333, 292, 0, 515, 495, 499, 488, 528, 489, 550, 488, 556, 551, 0, 0, 0, 0, 0, 0, 0, 657, 623, 584, 689, 568, 521, 672, 698, 276, 0, 0, 538, 826, 728, 734, 601, 760, 628, 577, 570, 700, 638, 910, 636, 634, 0, 0, 0, 0, 0, 0, 0, 515, 557, 441, 558, 513, 328, 550, 560, 249, 0, 0, 252, 852, 555

Loaded 1325 PDF pages from /home/ayush/Desktop/projects/rag-assistant/data


In [ ]:
# Extract PDF figures and graphs with PyMuPDF (does not require Pillow/PIL)
import hashlib
import sys
from pathlib import Path
print(f"Notebook kernel: {sys.executable}")
try:
    import pymupdf as fitz
except ImportError as exc:
    raise ImportError("PyMuPDF is missing from this notebook kernel. Select the project .venv kernel or run `%pip install pymupdf`, then restart the kernel.") from exc
from langchain_core.documents import Document

figure_dir = Path("/tmp/electric_motor_rag_figures")
figure_dir.mkdir(parents=True, exist_ok=True)
visual_chunks = []
visual_assets = []
visual_terms = ("graph", "chart", "curve", "plot", "characteristic", "diagram", "schematic")

for pdf_path in sorted(data_dir.rglob("*.pdf")):
    pdf = fitz.open(str(pdf_path))
    try:
        for page_index, page in enumerate(pdf):
            page_number = page_index + 1
            page_text = (page.get_text("text") or "").strip()
            detected_type = "graph" if any(term in page_text.lower() for term in visual_terms) else "image"
            page_visual_count = len(visual_chunks)

            # Extract embedded raster images directly as bytes; no PIL conversion.
            for image_number, image_info in enumerate(page.get_images(full=True), start=1):
                try:
                    extracted = pdf.extract_image(image_info[0])
                    image_bytes = extracted["image"]
                    image_suffix = "." + extracted.get("ext", "png")
                    digest = hashlib.sha256(
                        f"{pdf_path.resolve()}:{page_number}:{image_number}:".encode() + image_bytes
                    ).hexdigest()[:20]
                    image_path = figure_dir / f"{digest}{image_suffix}"
                    if not image_path.exists():
                        image_path.write_bytes(image_bytes)

                    caption_context = page_text[:4000] or "No selectable text was found on this PDF page."
                    search_text = (
                        f"Electric motor reference {detected_type} or figure. "
                        f"Source: {pdf_path.name}, page {page_number}. "
                        f"Page text and possible caption: {caption_context}"
                    )
                    metadata = {
                        "source": str(pdf_path.resolve()),
                        "page": page_index,
                        "visual_type": detected_type,
                        "image_path": str(image_path),
                        "figure_id": digest,
                    }
                    visual_chunks.append(Document(page_content=search_text, metadata=metadata))
                    visual_assets.append(metadata)
                except Exception as exc:
                    print(f"Skipped image {image_number} on {pdf_path.name}, page {page_number}: {exc}")

            # Some plots are vector drawings, so render those graph-like pages if no image was extracted.
            if len(visual_chunks) == page_visual_count and detected_type == "graph":
                try:
                    image_bytes = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5), alpha=False).tobytes("png")
                    digest = hashlib.sha256(
                        f"{pdf_path.resolve()}:{page_number}:vector-graph:".encode() + image_bytes
                    ).hexdigest()[:20]
                    image_path = figure_dir / f"{digest}.png"
                    if not image_path.exists():
                        image_path.write_bytes(image_bytes)
                    caption_context = page_text[:4000] or "Graph page without selectable text."
                    search_text = (
                        f"Electric motor graph or plotted curve. Source: {pdf_path.name}, "
                        f"page {page_number}. Page text and possible caption: {caption_context}"
                    )
                    metadata = {
                        "source": str(pdf_path.resolve()),
                        "page": page_index,
                        "visual_type": "graph",
                        "image_path": str(image_path),
                        "figure_id": digest,
                    }
                    visual_chunks.append(Document(page_content=search_text, metadata=metadata))
                    visual_assets.append(metadata)
                except Exception as exc:
                    print(f"Could not render graph page {pdf_path.name}, page {page_number}: {exc}")
    finally:
        pdf.close()

print(f"Extracted {len(visual_assets)} PDF figures/graphs to {figure_dir}")


In [ ]:
# creating chunks from text loaded
from langchain_text_splitters import RecursiveCharacterTextSplitter


def process_and_chunk(documents, chunk_size=1000, chunk_overlap=150):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} text chunks.")
    return split_docs


chunks = process_and_chunk(documents)
# Visual records use page text as their searchable description and retain the image path.
all_chunks = chunks + visual_chunks
print(f"Indexing {len(chunks)} text chunks and {len(visual_chunks)} visual records.")


In [3]:
# embedding
import os
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List, Any, Dict

project_root = Path.cwd()
if not (project_root / "data").is_dir() and (project_root.parent / "data").is_dir():
    project_root = project_root.parent
load_dotenv(project_root / ".env")

class EmbeddingManager:
    def __init__(self, model_name: str = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"loading embedding manager: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded with embedding dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"error loading model {self.model_name}: {e}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("model not loaded!")
        print(f"generating embedding for {len(texts)} chunks..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager


/home/ayush/Desktop/projects/rag-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading embedding manager: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2065.30it/s]


model loaded with embedding dimensions: 384


In [ ]:
# vector store
import os
from pathlib import Path
from uuid import NAMESPACE_URL, uuid5
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

project_root = Path.cwd()
if not (project_root / "data").is_dir() and (project_root.parent / "data").is_dir():
    project_root = project_root.parent
load_dotenv(project_root / ".env")

class VectorManager:
    def __init__(self, collection_name=None):
        qdrant_url = os.getenv("QDRANT_URL")
        qdrant_api_key = os.getenv("QDRANT_API_KEY")
        if not qdrant_url or not qdrant_api_key:
            raise ValueError("Set QDRANT_URL and QDRANT_API_KEY in the project .env file")

        self.collection_name = collection_name or os.getenv("QDRANT_COLLECTION", "electric_motors")
        self.client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
        if not self.client.collection_exists(self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=models.VectorParams(
                    size=embedding_manager.model.get_embedding_dimension(),
                    distance=models.Distance.COSINE,
                ),
            )
        print(f"Qdrant collection: {self.collection_name}")
        print(f"existing chunks in collection: {self.count()}")

    def count(self):
        return self.client.count(collection_name=self.collection_name, exact=True).count

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("number of documents does not match number of embeddings")

        points = []
        for doc, embedding in zip(documents, embeddings):
            source = str(doc.metadata.get("source", "unknown"))
            page = int(doc.metadata.get("page", 0))
            visual_id = doc.metadata.get("figure_id")
            stable_content_id = visual_id or doc.page_content
            point_id = str(uuid5(NAMESPACE_URL, f"{source}:{page}:{stable_content_id}"))
            payload = {
                "text": doc.page_content,
                "source": source,
                "page": page,
                "content_length": len(doc.page_content),
                "content_type": "visual" if visual_id else "text",
            }
            for key in ("visual_type", "image_path", "figure_id"):
                if key in doc.metadata:
                    payload[key] = doc.metadata[key]
            points.append(models.PointStruct(id=point_id, vector=embedding.tolist(), payload=payload))

        for start in range(0, len(points), 64):
            self.client.upsert(
                collection_name=self.collection_name,
                points=points[start:start + 64],
                wait=True,
            )
        print(f"uploaded {len(points)} text/visual records to Qdrant")
        print(f"total records in collection: {self.count()}")

vector_store = VectorManager()
vector_store


### PDF figures and plotted graphs

This cell uses PyMuPDF to extract embedded images and render vector-drawn graph pages without Pillow. Select the project `.venv` Python kernel in Codium/Jupyter before running. PyMuPDF is installed in that environment; if you use another kernel, install it there with `%pip install pymupdf` and restart. The cell prints the active kernel path. Visual page text/captions are indexed with the existing text embedding model; numeric values in plotted curves are not digitized automatically.


In [ ]:
# convert text and figure-page descriptions into embeddings
texts = [doc.page_content for doc in all_chunks]
embeddings = embedding_manager.generate_embedding(texts)
vector_store.add_documents(all_chunks, embeddings)


### retrieval pipeline

In [ ]:
class RagRetriever:
    def __init__(self, vector_store: VectorManager, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.30) -> List[Dict[str, Any]]:
        collection_count = self.vector_store.count()
        if collection_count == 0:
            return []

        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        results = self.vector_store.client.query_points(
            collection_name=self.vector_store.collection_name,
            query=query_embedding.tolist(),
            limit=min(top_k, collection_count),
            with_payload=True,
        )
        retrieved_docs = []
        for rank, point in enumerate(results.points, start=1):
            payload = point.payload or {}
            similarity_score = float(point.score or 0.0)
            if similarity_score >= score_threshold:
                retrieved_docs.append({
                    "id": str(point.id),
                    "content": str(payload.get("text", "")),
                    "metadata": {
                        "source": payload.get("source", "unknown"),
                        "page": payload.get("page", 0),
                        "content_type": payload.get("content_type", "text"),
                        "visual_type": payload.get("visual_type"),
                        "image_path": payload.get("image_path"),
                        "figure_id": payload.get("figure_id"),
                    },
                    "similarity_score": similarity_score,
                    "rank": rank,
                })
        return retrieved_docs

rag_retriever = RagRetriever(vector_store, embedding_manager)
rag_retriever


### vector store to llm output generation

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("GROQ_API"))

True

In [8]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

class GroqLLM:
    def __init__(self, model_name: str = os.getenv("GROQ_MODEL", "llama3-8b-8192"), api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a strict QA assistant. Answer the user's question using only the provided context. If the answer cannot be found in the context, reply exactly with: 'I do not have access to this information.' Do not use any outside knowledge.

Context:
{context}

Question: {question}
"""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
        

In [9]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    llm = None

Initialized Groq LLM with model: openai/gpt-oss-20b
Groq LLM initialized successfully!


In [10]:
rag_retriever.retrieve("what are synchronous machines")

generating embedding for 1 chunks..


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.05it/s]

generated embeddings with shape: (1, 384)


[{'id': '2b0d766d-d9e6-5c23-ad06-8dc89c947a6e',
  'content': '1552 Electrical Technology\nHence, these motors are used for synthetic-fibre drawing where constant speeds are absolutely essential.\n39.14. Synchros\nIt is a general name for self-synchronizing machines which, when electrically energized and\nelectrically interconnected, exert torques which cause two mechanically independent shafts either to\nrun in synchronism or to make the rotor of one unit follow the rotor position of the other. They are\nalso known by the trade names of selsyns and autosyns. Synchros, in fact, are small cylindrical motors\nvarying in diameter from 1.5 cm to 10 cm depending on their power output. They are low-torque\ndevices and are widely used in control systems for transmitting shaft position information or for\nmaking two or more shafts to run in synchronism. If a large device like a robot arm is to be positioned,\nsynchros will not work. Usually, a servomotor is needed for a higher torque.\n39.15. T

In [ ]:
# llm integration
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
import os
import re
import time
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API") or os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key, model_name=os.getenv("GROQ_MODEL", "llama3-8b-8192"), temperature=0.1, max_tokens=1024)
NO_ANSWER = "I couldn't find that in the electric motor reference material."
SYSTEM_PROMPT = """You are an intelligent AI assistant powered by a
Retrieval-Augmented Generation (RAG) system.

YOUR MAIN RESPONSIBILITY:
Understand the user's actual intention and provide
accurate, helpful, natural, and easy-to-understand answers.

1. UNDERSTANDING USER QUESTIONS
- Understand simple, casual and informal language.
- Do not require users to use technical terminology.
- Understand spelling mistakes and grammatical errors.
- Understand incomplete and poorly structured questions.
- Identify the actual meaning behind the question.
- Understand synonyms and alternative expressions.
- Do not depend on exact keyword matching.
- Interpret what the user means, not just what they type.

2. QUERY INTERPRETATION
- Convert unclear questions into meaningful search queries.
- Identify important concepts and related terminology.
- Consider different possible meanings of a question.
- Search for relevant information even when the user's
  wording differs from the document's wording.
- Never change the original intent of the question.

3. CONTEXT AWARENESS
- Remember previous questions in the conversation.
- Understand follow-up questions such as:
  "Explain more", "Why?", "How does it work?",
  "Give an example", and "What about its types?"
- Connect follow-up questions to the previous topic.
- Do not ask users to repeat information unnecessarily.

4. DOCUMENT RETRIEVAL
- Use the retrieved document context carefully.
- Consider all relevant retrieved passages.
- Combine information from multiple relevant passages.
- Do not select information based only on keyword matches.
- Ignore irrelevant retrieved content.
- If information is missing, clearly acknowledge it.

5. ANSWER GENERATION
- Answer the exact question asked.
- Use simple and understandable English.
- Explain technical terms in ordinary language.
- Prefer clear and direct answers.
- Break complicated concepts into smaller parts.
- Provide examples wherever useful.
- Use bullet points and headings when appropriate.
- Avoid unnecessary technical jargon.
- Do not give unnecessarily long answers.

6. ACCURACY AND HONESTY
- Never invent facts, figures, or references.
- Do not assume information that is not provided.
- Distinguish document-based facts from general knowledge.
- If the retrieved context does not contain the answer,
  clearly say that the documents do not provide it.
- Never pretend to have found information that is missing.

7. CONVERSATIONAL BEHAVIOUR
- Be friendly, natural and helpful.
- Respond like a human tutor.
- Do not sound robotic.
- Handle greetings and casual conversation naturally.
- Ask a clarification question only when necessary.
- If the question is understandable, answer directly.

8. EXPLANATION STYLE
When explaining a topic:
- Start with a simple definition.
- Explain how it works.
- Give a practical example.
- Explain important types or applications if relevant.
- Provide a short summary when useful.

9. OUTPUT QUALITY
- Prioritize relevance over answer length.
- Do not repeat the same information.
- Use examples related to the question.
- Make answers understandable even to beginners.

IMPORTANT:
Your primary purpose is to help users understand
information, not simply repeat retrieved document text."""

def format_bare_math(text):
    text = re.sub(
        r"\\\[(.*?)\\\]",
        lambda match: "$$\n" + match.group(1).strip() + "\n$$",
        text,
        flags=re.DOTALL,
    )
    text = re.sub(
        r"\\\((.*?)\\\)",
        lambda match: "$" + match.group(1).strip() + "$",
        text,
        flags=re.DOTALL,
    )

    def wrap_bare_formula(match):
        formula = match.group(1).strip().rstrip(" ;,.")
        return f"${formula}$"

    return re.sub(
        r"(?<!\\)\[\s*([^\[\]\n]*?(?:\\[A-Za-z]+|=)[^\[\]\n]*?)\s*(?<!\\)\]",
        wrap_bare_formula,
        text,
    )

def rag_simple(query, retriever, llm, top_k=5, min_score=0.30, return_visuals=False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return (NO_ANSWER, []) if return_visuals else NO_ANSWER

    context = "\n\n".join(
        f"[{index}] {doc['content']}" for index, doc in enumerate(results, start=1)
    )
    prompt = f"""Use the following retrieved passages to answer the question.
If no passage contains the answer, reply exactly: {NO_ANSWER}
Include the supporting passage number, such as [1].
Write equations in LaTeX using $...$ inline or $$...$$ on separate lines; do not use bare square brackets as math delimiters.

Passages:
{context}

Question: {query}
Answer:"""
    for attempt in range(3):
        try:
            response = llm.invoke([SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=prompt)])
            break
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    answer = response.content
    if not isinstance(answer, str):
        answer = "".join(str(part) for part in answer)
    answer = answer.strip()
    answer = format_bare_math(answer)
    visuals = []
    seen_visuals = set()
    for result in results:
        metadata = result["metadata"]
        image_path = metadata.get("image_path")
        if metadata.get("content_type") == "visual" and image_path and image_path not in seen_visuals:
            visuals.append({"path": image_path, "type": metadata.get("visual_type") or "image",
                            "source": metadata.get("source"), "page": metadata.get("page", 0) + 1})
            seen_visuals.add(image_path)
        if len(visuals) >= 3:
            break
    return (answer, visuals) if return_visuals else answer


def display_rag_answer(answer, visuals):
    from IPython.display import Markdown, display, Image as NotebookImage
    from pathlib import Path

    display(Markdown(answer))
    for visual in visuals:
        image_path = Path(visual["path"])
        if not image_path.is_file():
            continue
        source_name = Path(visual["source"]).name if visual.get("source") else "PDF reference"
        visual_label = "Graph/plot" if visual["type"] == "graph" else "Figure/image"
        display(Markdown(f"**{visual_label} — {source_name}, page {visual['page']}**"))
        display(NotebookImage(filename=str(image_path)))


In [ ]:
answer, related_visuals = rag_simple(
    "what is a synchronous motor", rag_retriever, llm, return_visuals=True
)
display_rag_answer(answer, related_visuals)
